# Guided Solver Config Builder

Configure your model inputs and write `solver_params.json` to disk.
Run this notebook before `guided_bayesian_inference.ipynb`.

In [2]:
import sys
sys.path.append("../")
from solver_config_builder import build_solver_params_config, write_solver_params_json_file

In [3]:
# =======================
# FILE PATHS
# =======================
# Use either a single file ("./reactions_MA.json") or a folder ("./rxnfiles")
reactions_source = "./Reactions"

# Set save directory and output file names
savedir = "./Results"
solver_params_file = "solver_params.json"
prior_samples_file = "MA_prior_samples_pm.nc"
posterior_samples_file = "MA_posterior_samples_pm.nc"
trace_plot_file = "trace_plot.png"

# =============================
# FREE PARAMETERS
# =============================
# Required fields: rxn_name, param_name, distribution, lower, upper
# Optional fields: mass (default: 0.95), fixed_stat
free_parameter_prior_inputs = [
    {"rxn_name": "binding", "param_name": "k_f", "distribution": "Gamma", "lower": 1.0e-2, "upper": 2.0e-1, "mass": 0.95},
    {"rxn_name": "binding", "param_name": "k_r", "distribution": "Gamma", "lower": 2.5e-3, "upper": 5.0e-2, "mass": 0.95},
    {"rxn_name": "catalysis", "param_name": "k_cat", "distribution": "Gamma", "lower": 1.0e-1, "upper": 2.0, "mass": 0.95},
]

# ========================
# DATASETS
# ========================
# Replace this with your preferred dataset set.
dataset_inputs = [
    {
        "name": "pyruvate_kinase_product_fraction",
        "dataset_type": "timeseries",
        "data_file": "./Data/Duggleby_Clarke_1991_Fig2.csv",
        "observable": "product_fraction",
        "time_column": "time",
        "column_mapping": {"product_fraction": "[P]/[S_tot]"},
        "noise_model": "relative_mean",
        "noise_params": {"frac": 0.05},
    }
]

# =====================================
# SAMPLING + SOLVER SETTINGS
# =====================================
prior_sampling_settings = {"draws": 10000, "random_seed": 0}
posterior_sampling_settings = {
    "draws": 100,
    "tune": 100,
    "chains": 4,
    "cores": "None",
    "random_seed": 0,
    "nuts_sampler": "nutpie",
}

ode_solver_settings = {
    "solver_name": "Kvaerno5",
    "dt0": 1.0e-12,
    "max_steps": 10000000,
    "stepsize_controller": "PIDController",
}

ode_stepsize_controller_settings = {
    "rtol": 1.0e-9,
    "atol": 1.0e-9,
    "pcoeff": 0.3,
    "icoeff": 0.4,
    "dcoeff": 0.0,
}

# =================================
# MODEL-SPECIFIC SETTINGS
# =================================
calculation_module_path = "./pyruvate_kinase_calculations.py"
initial_conditions = {"S": 0.195, "E": 1.0}

In [4]:
solver_params = build_solver_params_config(
    free_parameter_prior_inputs=free_parameter_prior_inputs,
    dataset_inputs=dataset_inputs,
    prior_sampling_settings=prior_sampling_settings,
    posterior_sampling_settings=posterior_sampling_settings,
    ode_solver_settings=ode_solver_settings,
    ode_stepsize_controller_settings=ode_stepsize_controller_settings,
    calculation_module_path=calculation_module_path,
    initial_conditions=initial_conditions,
)

written_solver_params_path = write_solver_params_json_file(
    solver_params_config=solver_params,
    file_directory=".",
    filename=solver_params_file,
)

print(f"Wrote solver config: {written_solver_params_path}")
print(f"Reaction source: {reactions_source}")
print(f"Save directory: {savedir}")
print(
    f"Configured free params: {[param['param_name'] for param in solver_params['free_kinetic_params']]}"
)
print(
    f"Configured datasets: {[dataset['name'] for dataset in solver_params['datasets']]}"
)

Wrote solver config: /Users/annettethompson/Library/CloudStorage/OneDrive-SharedLibraries-UCB-O365/Jerome Michael Fox - Annie Thompson/Git Repositories/Bayesian Kinetic Model/Restructured Framework/MA_Model/solver_params.json
Reaction source: ./Reactions
Save directory: ./Results
Configured free params: ['k_f', 'k_r', 'k_cat']
Configured datasets: ['pyruvate_kinase_product_fraction']
